# Yantra — DPO Round-2: train on 66 arg-failures → export → re-eval

Upload to Colab (T4 GPU) → **Runtime → Run all**. One session, ~45 min total:

1. **Cell 3 (~15–25 min, GPU):** DPO on `dpo_round2_from_eval.jsonl` (66 pairs, all `arg_value`) starting from the same adapters lineage as your 0.6746 GGUF. Same `DPOTrainer` config as pipeline Stage 4, but 3 epochs (66 pairs ≈ 8 steps/epoch — 1 epoch would barely move weights).
2. **Cell 4 (~10–15 min):** merge + export Q4_K_M GGUF, mirror to Drive. Frees GPU afterwards.
3. **Cell 5 (~2 min):** re-eval with the identical Router V2 + fixed-stops harness. Target **PAS 0.70+** (`exact_args` 0.65 → 0.75+).

Inputs required on Drive (all present from your last run): `yantra_results_router_v2_stopsfixed.json` (0.6746), `dpo_round2_from_eval.jsonl` (66 pairs), `stage5_rte/adapters` (or `stage4_egopd/adapters` fallback). Nothing here touches main-pipeline markers.

In [ ]:
# @title 0 — Mount Drive & check prerequisites (no GPU needed)
import os, sys, json, re, math, shutil, time, subprocess
from pathlib import Path
from collections import Counter

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_DIR = Path('/content/drive/MyDrive/yantra_run')
except Exception as e:
    RUN_DIR = Path('./yantra_run')
    print('Drive not mounted:', e, '→ using', RUN_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)
ART = RUN_DIR / 'artifacts'
print('RUN_DIR =', RUN_DIR)
print('ART =', ART)

# Prereqs from the 0.6746 run
for f in ['toolace_300.jsonl', 'dpo_round2_from_eval.jsonl',
          'yantra_results_router_v2_stopsfixed.json']:
    p = ART / f
    print(('OK   ' if p.exists() else 'MISS ') + f'  {p}'
          + (f' ({p.stat().st_size/1024:.0f} KB)' if p.exists() else ''))
pairs = [json.loads(l) for l in open(ART/'dpo_round2_from_eval.jsonl') if l.strip()] \
    if (ART/'dpo_round2_from_eval.jsonl').exists() else []
print(f'pairs: {len(pairs)} (expect 66)')
if pairs:
    print('pair keys:', sorted(pairs[0].keys()))

# Base adapters: same lineage as the evaluated GGUF (stage5 preferred, else stage4/3)
_cands = [ART/'stage5_rte'/'adapters', ART/'stage4_egopd'/'adapters',
          ART/'stage3_dtsa_sft'/'adapters']
BASE = next((p for p in _cands if p.exists()), None)
print('BASE adapters =', BASE if BASE else 'NONE FOUND → train cell will error clearly')


In [ ]:
# @title 1 — Installs: unsloth (training) + llama-cpp-python (eval). ~5 min.
import subprocess, sys
def _sh(cmd):
    return subprocess.run(cmd, shell=True).returncode == 0

# Unsloth: same unpinned logic as main pipeline Stage 0 (never pin xformers/torch).
try:
    import unsloth  # noqa: F401
    print('unsloth already importable')
except Exception:
    print('installing unsloth ...')
    if not _sh('pip install -q --upgrade unsloth unsloth_zoo'):
        print('PyPI failed; trying git colab-new build ...')
        _sh('pip install -q --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')
    _sh('pip install -q sentencepiece protobuf datasets huggingface_hub hf_transfer')
    import unsloth
    print('unsloth installed')

try:
    import llama_cpp
    print('llama_cpp', llama_cpp.__version__)
except Exception:
    print('installing llama-cpp-python ...')
    if _sh(f'{sys.executable} -m pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121'):
        import llama_cpp
    else:
        _sh(f'{sys.executable} -m pip install -q llama-cpp-python')
        import llama_cpp
    print('llama_cpp', llama_cpp.__version__)

import torch
HAS_CUDA = torch.cuda.is_available()
print('HAS_CUDA =', HAS_CUDA)
if HAS_CUDA:
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No CUDA — DPO training needs a Colab T4 (Runtime → Change runtime type → T4 GPU).')


In [ ]:
# @title 2 — Shared utils (Router V2 + parsers + metrics, same as 0.6746 run)
import json, re, math
from collections import Counter

ACTION_END = '<action_end/>'
DTSA_BIND = re.compile(r'<bind\s+tool="([^"]+)"\s*/>', re.S)
DTSA_ARGS = re.compile(r'<args>(.*?)</args>', re.S)
DTSA_PARAM = re.compile(r'<param\s+name="([^"]+)"\s*>(.*?)</param>', re.S)
def _unesc(v): return v.replace('&amp;','&').replace('&lt;','<').replace('&gt;','>')
def to_dtsa(name, arguments):
    lines = ['<bind tool="%s"/>' % name, '<args>']
    for k,v in arguments.items():
        v = '' if v is None else str(v)
        v = v.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')
        lines.append('  <param name="%s">%s</param>' % (k,v))
    lines += ['</args>', ACTION_END]
    return '\n'.join(lines)
def bind_prefix(name): return f'<bind tool="{name}"/>\n'
def dtsa_args_block(name, args): return '\n'.join(to_dtsa(name,args).splitlines()[1:])
def strip_bind(text):
    m = DTSA_BIND.search(text)
    return text[m.end():] if m else text

def parse_dtsa(text):
    binds = list(DTSA_BIND.finditer(text))
    if not binds: return None
    ends = list(re.finditer(re.escape(ACTION_END), text))
    stopped = bool(ends) and text[ends[-1].end():].strip() == ''
    for j in range(len(binds)-1, -1, -1):
        lb = binds[j]
        seg_end = binds[j+1].start() if j+1 < len(binds) else len(text)
        am = DTSA_ARGS.search(text[lb.end():seg_end])
        if not am: continue
        body = am.group(1); args = {}
        for pm in DTSA_PARAM.finditer(body):
            args[pm.group(1)] = _unesc(pm.group(2).strip())
        if not args:
            for lm in re.finditer(r'name="([^"]+)"\s*>[ \t]*(.*)', body):
                args[lm.group(1)] = _unesc(lm.group(2).strip())
        return {'tool': lb.group(1), 'args': args, 'stopped_clean': stopped}
    return {'tool': binds[0].group(1), 'args': {}, 'stopped_clean': stopped}

def _norm(a):
    if isinstance(a, str):
        s = a.strip()
        if s.lower() in ('true','false'): return s.lower()=='true'
        try: return int(s) if '.' not in s else float(s)
        except ValueError: return s
    return a
def avail_names(tools): return {t.get('function', t).get('name') for t in tools}
def evaluate_case(text, case, mode='dtsa'):
    tools = case['tools']; gold = case['gold']; gold0 = gold[0]
    parsed = parse_dtsa(text)
    if not parsed: return {'parseable':0,'valid_name':0,'expected_name':0,'exact_args':0,'arg_key_overlap':0,'stopped_cleanly':0}
    valid = parsed['tool'] in avail_names(tools)
    expected = parsed['tool'] == gold0['name']
    gk = set(gold0['arguments'].keys()); pk = set(parsed['args'].keys())
    overlap = len(gk & pk)/len(gk) if gk else 1.0
    exact = (set(parsed['args'].keys()) == gk) and all(_norm(gold0['arguments'][k]) == _norm(parsed['args'][k]) for k in gk) if gk else bool(parsed)
    return {'parseable':1,'valid_name':int(valid),'expected_name':int(expected),'exact_args':int(exact),'arg_key_overlap':round(overlap,4),'stopped_cleanly':int(parsed['stopped_clean'])}
def pas(summary, recovery=0.0, multiturn=0.0):
    comps = [summary.get(k,0.0) for k in ('parseable','valid_name','expected_name','exact_args','arg_key_overlap','stopped_cleanly')] + [recovery, multiturn]
    return round(sum(comps)/len(comps), 4)

STOP = ['\n<bind', '<tool_result>', '<user>', '</calls>', '<tool_error>', '<reflect>']
print('utils loaded. STOP =', STOP)


In [ ]:
# @title 3 — DPO Round-2 training (~15-25 min on T4 for 66 pairs x 3 epochs)
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
from unsloth import FastLanguageModel
from trl import DPOTrainer, DPOConfig
from datasets import Dataset

if BASE is None or not Path(BASE).exists():
    raise FileNotFoundError('No base adapters found (checked stage5/stage4/stage3). Train the main pipeline first.')
print('Base:', BASE)

pairs_all = [json.loads(l) for l in open(ART/'dpo_round2_from_eval.jsonl') if l.strip()]
print(f'Loaded {len(pairs_all)} pairs')
assert len(pairs_all) >= 10, f'Too few pairs ({len(pairs_all)}) — DPO would be noise.'
# DPOTrainer only needs prompt/chosen/rejected; drop helper keys.
train_rows = [{'prompt': p['prompt'], 'chosen': p['chosen'], 'rejected': p['rejected']}
              for p in pairs_all]
ds = Dataset.from_list(train_rows)
print('Train sample prompt len:', len(train_rows[0]['prompt']),
      '| chosen len:', len(train_rows[0]['chosen']),
      '| rejected len:', len(train_rows[0]['rejected']))

out = ART/'stage6_dpo2_round2'
out.mkdir(parents=True, exist_ok=True)

# Same load pattern as pipeline Stage 4 fresh-start branch (adapters dir + same LoRA config).
model, tokenizer = FastLanguageModel.from_pretrained(str(BASE), max_seq_length=2048)
model.generation_config.max_length = None
model = FastLanguageModel.get_peft_model(
    model, r=64, lora_alpha=128, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

# 3 epochs: 66 pairs at eff. batch 8 ≈ 8 steps/epoch → ~25 steps total.
trainer = DPOTrainer(
    model=model, ref_model=None, tokenizer=tokenizer, train_dataset=ds,
    args=DPOConfig(per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_ratio=0.03, num_train_epochs=3, learning_rate=5e-5, fp16=True,
        logging_steps=5, save_strategy="no", output_dir=str(out)))
trainer.train()
model.save_pretrained(str(out/"adapters"))
tokenizer.save_pretrained(str(out/"adapters"))
print('DPO Round-2 done →', out/"adapters")


In [ ]:
# @title 4 — Merge + export Q4_K_M GGUF (~10-15 min), then free GPU
import shutil, gc
try:
    import torch
except Exception:
    pass

# If kernel restarted after Cell 3, reload the round-2 adapters.
try:
    model
    print('Using in-memory trained model')
except NameError:
    from unsloth import FastLanguageModel
    _ad = ART/'stage6_dpo2_round2'/'adapters'
    if not _ad.exists():
        raise FileNotFoundError(f'{_ad} not found — run Cell 3 first.')
    print('Reloading', _ad)
    model, tokenizer = FastLanguageModel.from_pretrained(str(_ad), max_seq_length=2048)

for d in (Path('/content/stage7_dpo2_q4'), Path('/content/stage7_dpo2_q4_gguf')):
    if d.exists():
        shutil.rmtree(d)
print('Exporting Q4_K_M (internal 16-bit merge, single call) ...')
model.save_pretrained_gguf('/content/stage7_dpo2_q4', tokenizer,
                            quantization_method='q4_k_m')

def _find_new_gguf():
    for d in (Path('/content/stage7_dpo2_q4_gguf'), Path('/content/stage7_dpo2_q4')):
        if d.exists():
            g = sorted(d.glob('*Q4_K_M*.gguf')) or sorted(d.glob('*.gguf'))
            if g:
                return g[0]
    return None

NEW_GGUF = _find_new_gguf()
assert NEW_GGUF is not None, 'GGUF export produced no file'
print(f'Exported: {NEW_GGUF} ({NEW_GGUF.stat().st_size/1024/1024:.0f} MB)')

try:
    dst = ART/'stage7_dpo2_q4_gguf'
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(str(NEW_GGUF.parent), str(dst))
    print('Mirrored to Drive:', dst)
except Exception as e:
    print('Drive mirror skipped (non-fatal, eval continues):', e)

# Free GPU so llama-cpp can use it in Cell 5.
for _n in ('trainer', 'model', 'tokenizer'):
    try:
        del globals()[_n]
    except KeyError:
        pass
gc.collect()
try:
    torch.cuda.empty_cache()
except Exception:
    pass
print('GPU freed. NEW_GGUF =', NEW_GGUF)


In [ ]:
# @title 5 — Re-eval new GGUF (Router V2 + fixed stops, ~2 min) + compare vs 0.6746
from llama_cpp import Llama
import time

# Prefer local export; fall back to Drive mirror (e.g. after a restart).
_gg = None
try:
    _gg = NEW_GGUF  # noqa: F821 — set in Cell 4
except NameError:
    _gg = None
if _gg is None or not Path(str(_gg)).exists():
    for d in (Path('/content/stage7_dpo2_q4_gguf'), Path('/content/stage7_dpo2_q4'),
              ART/'stage7_dpo2_q4_gguf'):
        if d.exists():
            g = sorted(d.glob('*Q4_K_M*.gguf')) or sorted(d.glob('*.gguf'))
            if g:
                _gg = g[0]; break
if _gg is None:
    raise FileNotFoundError('No DPO2 GGUF found — run Cell 4 first.')
print('Evaluating:', _gg)
llm2 = Llama(model_path=str(_gg), n_ctx=4096,
             n_gpu_layers=(-1 if HAS_CUDA else 0), verbose=False)
print('Model loaded.')

cases = [json.loads(l) for l in open(ART/'toolace_300.jsonl') if l.strip()]
toks = lambda s: set(re.findall(r'[a-z0-9_]+', s.lower()))
_docs = [toks((t.get('function',t).get('name','')+' '+t.get('function',t).get('description',''))) for c in cases for t in c['tools']]
_df = Counter(w for d in _docs for w in d); _N = max(len(_docs),1)
_idf = lambda w: math.log(_N/(1+_df[w]))
def cgrams(s, n=3):
    s = re.sub(r'[^a-z0-9 ]','',s.lower())
    return {s[i:i+n] for i in range(max(len(s)-n+1,1))}
def route(q, tools):
    qt = toks(q); qg = cgrams(q)
    best, bn = -1e18, None
    for t in tools:
        f = t.get('function', t); nm = f.get('name') or ''
        d = toks(nm+' '+(f.get('description') or ''))
        s1 = sum(_idf(w) for w in qt & d)/(math.sqrt(sum(_idf(w) for w in d)+1e-6))
        ng = cgrams(nm)
        s2 = len(qg & ng)/(math.sqrt(len(qg)*len(ng))+1)
        if s1 + 2.0*s2 > best: best, bn = s1 + 2.0*s2, nm
    return bn

per = []
t0 = time.time()
for i, case in enumerate(cases):
    bind = f'<bind tool="{route(case["query"], case["tools"])}"/>\n'
    prompt = f"<user>{case['query']}</user>\n<tools>{json.dumps([t.get('function',t) for t in case['tools']], ensure_ascii=False)}</tools>\n<calls>{bind}"
    out = llm2(prompt, max_tokens=768, temperature=0.0, stop=STOP)
    text = out['choices'][0]['text'] or ''
    per.append({'id': i, **evaluate_case(bind + text, case)})

    if (i+1) % 50 == 0 or i == len(cases)-1:
        agg_t = {}
        for mm in per:
            for k, v in mm.items():
                if k == 'id': continue
                agg_t[k] = agg_t.get(k, 0.0) + v
        st = {k: round(v/len(per), 4) for k, v in agg_t.items()}
        print(f"  eval {i+1}/{len(cases)}  pas={pas(st):.4f}  exact={st.get('exact_args',0):.3f}  {time.time()-t0:.0f}s", flush=True)

agg = {}
for m in per:
    for k, v in m.items():
        if k == 'id': continue
        agg[k] = agg.get(k, 0.0) + v
summary = {k: round(v/len(cases), 4) for k, v in agg.items()}
score = pas(summary)
(ART/'yantra_results_dpo2.json').write_text(
    json.dumps({'summary': summary, 'pas': score, 'per_case': per}, indent=2))
old = json.loads((ART/'yantra_results_router_v2_stopsfixed.json').read_text())
print('\n' + '='*50)
print(f"BEFORE (stops-fixed): PAS = {old['pas']}  {old['summary']}")
print(f"AFTER  (DPO round-2): PAS = {score}  {summary}")
print(f"Δ exact_args: {old['summary']['exact_args']:.4f} → {summary['exact_args']:.4f} "
      f"({summary['exact_args']-old['summary']['exact_args']:+.4f})")
print(f"Saved to {ART/'yantra_results_dpo2.json'}")
print('If improved: promote with  !cp .../yantra_results_dpo2.json .../yantra_results.json')
